# Phase 2 — Leakage experiment

**Question:** does the ~93% accuracy reported by Esen et al. (IEEE Access, 2025) come from the model, or from how the data was split?

Same model, same images, trained three times. Only the cross-validation split changes:

| Protocol | What goes into the test fold |
|---|---|
| `random` (paper) | random augmented copies, so rotated twins of test images sit in the training set |
| `image` | all copies of an original mammogram together |
| `patient` | all mammograms of a patient together |

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings → **Internet on**.
Attach: *Breast mammography images with Masses* (tommyngx) and *INbreast Dataset* (ramanathansp20).
Then **Run All**. Takes ~40 minutes.

In [ ]:
# 1) Get the code and check the machine
import os, subprocess, sys, torch
REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on Settings -> Accelerator -> GPU")
assert torch.cuda.is_available(), "Turn on the GPU: Settings -> Accelerator -> GPU T4 x2, then Run All again."
os.environ["PYTHONPATH"] = f"{REPO}/src"


In [ ]:
# 2) Unit tests + 2-minute smoke run (tiny subset). If this cell fails, stop and send Claude a screenshot.
!cd /kaggle/working/repo && python -m pytest -q tests 2>&1 | tail -3
!cd /kaggle/working/repo && python -m mammo.experiments.leakage --smoke


In [ ]:
# 3) Full experiment (~40 min). Progress is printed per epoch.
!cd /kaggle/working/repo && python -m mammo.experiments.leakage


In [ ]:
# 4) Results
import pandas as pd
from IPython.display import Image, display
OUT = "/kaggle/working/results/leakage"
display(Image(f"{OUT}/leakage_chart.png"))
s = pd.read_csv(f"{OUT}/summary.csv", index_col=0)
display(s[[c for c in s.columns if c.endswith("_mean")]].round(3))


In [ ]:
# 5) Pack everything for Claude -> download leakage_results.zip from Output (right panel, /kaggle/working)
!cd /kaggle/working && zip -qr leakage_results.zip results/leakage && ls -lh leakage_results.zip
